# LELA60341: Research Methods in Computational and Corpus Linguistics Week 10

## A text classifier, built from parts

In Week 8 you built a logistic regression model out of parts: `Parameter`, a `Linear` layer, a loss, an optimiser, and a model class that puts them together. In Week 9 you wrote `TextEncoder`, which turns texts into sparse one-hot rows. This week you use them together to classify the Yelp reviews: the classifier from the LELA60331 Week 9 seminar.

Only one new part is needed: a **sparse** version of the `Linear` layer, which works with one-hot rows stored as lists of columns. Everything else (the loss, the optimiser, the training loop, the evaluation functions) is reused exactly as it is.

```python
encoder = TextEncoder(5000)
encoder.fit_vocabulary(texts)
M_train = encoder.one_hot_encode(train_texts)

model = TextLogisticRegression(5000)
loss_fn = BCEWithLogitsLoss()
optimizer = SGD(model.parameters(), lr=0.4)
# ... the same training loop as in Week 8 ...
print(evaluate(model, encoder.one_hot_encode(test_texts), test_labels))
```

| Part | Topic |
|---|---|
| 0 | Set-up: the data, `TextEncoder`, and the parts from Week 8 |
| 1 | A sparse linear layer |
| 2 | Training and evaluating on the Yelp reviews |
| 3 | Inspecting the weights |
| 4 | A better tokeniser: just swap the encoder |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

Run the next four cells. They download the reviews, and set up:

* the data (`load_reviews` and the training and test split) and the `TextEncoder` class, from Week 9
* the helper functions `dot`, `mat_vec` and `transpose`, and the **safer** `sigmoid` from LELA60331 Week 9, which doesn't fail for very large negative scores
* the parts from Week 8: `zeros_like`, `Parameter`, `Module`, `Linear`, `SGD`, `BCEWithLogitsLoss`, and the functions `predict_proba`, `predict` and `evaluate`

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = line.rstrip().split("\t")
        texts.append(fields[0])
        labels.append(int(fields[1] == "positive"))
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")

random.seed(1)                          # the same split as in Week 9 and in LELA60331
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]
train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]


class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = []
        self._word_to_index = {}

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = counts.get(word, 0) + 1
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = [word for count, word in pairs[:self._vocab_size]]
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            self._word_to_index[self._type_list[j]] = j

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if word in self._word_to_index:
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return [self[j] for j in row]

print(len(train_texts), "training and", len(test_texts), "test reviews")

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

In [ ]:
def zeros_like(value):
    # a list or matrix of 0s with the same shape as value
    if isinstance(value[0], list):
        return [[0] * len(row) for row in value]
    else:
        return [0] * len(value)

class Parameter:
    """Numbers that training changes: their values, and their gradients."""
    def __init__(self, value):
        self.value = value
        self.grad = zeros_like(value)

class Module:
    """The parent class of every layer and every model."""
    def __init__(self):
        pass

    def __call__(self, X):
        return self.forward(X)

    def parameters(self):
        params = []
        for value in vars(self).values():
            if isinstance(value, Parameter):
                params.append(value)
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params

class Linear(Module):
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = Parameter([[random.random() for j in range(n_in)] for c in range(n_out)])
        self.bias = Parameter([random.random() for c in range(n_out)])

    def forward(self, X):
        self._X = X
        out = []
        for x in X:
            z = mat_vec(self.weight.value, x)
            for c in range(len(z)):
                z[c] += self.bias.value[c]
            out.append(z)
        return out

    def backward(self, grad_out):
        W = self.weight.value
        for k in range(len(self._X)):
            for c in range(len(W)):
                for j in range(len(W[c])):
                    self.weight.grad[c][j] += grad_out[k][c] * self._X[k][j]
                self.bias.grad[c] += grad_out[k][c]
        WT = transpose(W)
        return [mat_vec(WT, grad_out[k]) for k in range(len(self._X))]

class SGD:
    """Gradient descent: new value = old value - learning rate * gradient."""
    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = zeros_like(p.value)

    def step(self):
        for p in self.parameters:
            if isinstance(p.value[0], list):
                for i in range(len(p.value)):
                    for j in range(len(p.value[i])):
                        p.value[i][j] -= self.lr * p.grad[i][j]
            else:
                for i in range(len(p.value)):
                    p.value[i] -= self.lr * p.grad[i]

class BCEWithLogitsLoss:
    """The cross-entropy loss for two classes, from the scores."""
    def __call__(self, scores, y):
        self._q = [sigmoid(row[0]) for row in scores]
        self._y = y
        eps = 0.00001
        losses = []
        for k in range(len(y)):
            if y[k] == 1:
                losses.append(-math.log2(self._q[k] + eps))
            else:
                losses.append(-math.log2(1 - self._q[k] + eps))
        return sum(losses) / len(losses)

    def grad(self):
        n = len(self._y)
        return [[(self._q[k] - self._y[k]) / n] for k in range(n)]

def predict_proba(model, X):
    return [sigmoid(row[0]) for row in model(X)]

def predict(model, X):
    return [int(p > 0.5) for p in predict_proba(model, X)]

def evaluate(model, X, y):
    predicted = predict(model, X)
    correct = 0
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(y)):
        if predicted[k] == y[k]:
            correct += 1
        if predicted[k] == 1 and y[k] == 1:
            tp += 1
        if predicted[k] == 1 and y[k] == 0:
            fp += 1
        if predicted[k] == 0 and y[k] == 1:
            fn += 1
    return {"accuracy": correct / len(y), "precision": tp / (tp + fp), "recall": tp / (tp + fn)}

---
# Part 1 · A sparse linear layer

Each review is a one-hot row over 5000 vocabulary words, stored **sparsely**: as the list of the columns that hold a 1 (Week 9, and LELA60331 Week 8). The `Linear` layer from Week 8 expects full rows of numbers, and storing 8000 full rows of 5000 numbers would be far too slow. So we write a subclass, `SparseLinear`, that does the same job for sparse rows. It **inherits** the constructor (the weights and biases) from `Linear`, and **overrides** `forward` and `backward` (Week 7, Part 4).

**`forward(M)`.** For each sparse row, the score of each output `c` is its bias plus the sum of its weights at the columns that hold a 1 (LELA60331 Week 8, Activity 8).

**`backward(grad_out)`.** For each example `k` and each output `c`, the gradient of `weight[c][j]` gets `grad_out[k][c]` added **for each column `j` in the example's row**, because the feature value there is 1 (for every other column it is 0, so nothing is added). The bias gets `grad_out[k][c]`, as before (LELA60331 Week 8, Activity 9). The inputs are words, not numbers, so there are no gradients for them: `backward` returns nothing.

### 🏋️ Exercise 1 · Practice — `SparseLinear`

Fill in the blanks 1–4. The test underneath checks that `SparseLinear` gives exactly the same scores and gradients as `Linear` with the same weights, given the same examples written out in full.

In [ ]:
class SparseLinear(Linear):
    """A Linear layer for sparse one-hot rows (lists of the columns that hold a 1)."""
    def forward(self, M):
        self._X = M
        W = self.weight.value
        out = []
        for row in M:
            z = []
            for c in range(len(W)):
                total = ???            # 1. start with the bias
                for j in row:
                    total += ???                  # 2. add the weight of each column that holds a 1
                z.append(total)
            out.append(z)
        return out

    def backward(self, grad_out):
        for k in range(len(self._X)):
            for c in range(len(self.weight.value)):
                for j in self._X[k]:
                    self.weight.grad[c][j] += ???      # 3.
                self.bias.grad[c] += ???               # 4.


random.seed(1)
sparse = SparseLinear(4, 2)
random.seed(1)
dense = Linear(4, 2)                         # the same random weights
sparse_rows = [[0, 2], [1, 2, 3]]
full_rows = [[1, 0, 1, 0], [0, 1, 1, 1]]     # the same examples, written out in full
print(sparse(sparse_rows))
print(dense(full_rows))                      # should be the same

In [ ]:
# ✅ Check
def _same():
    random.seed(1); s = SparseLinear(4, 2)
    random.seed(1); d = Linear(4, 2)
    a, b = s([[0, 2], [1, 2, 3]]), d([[1, 0, 1, 0], [0, 1, 1, 1]])
    return all(abs(a[k][c] - b[k][c]) < 1e-12 for k in range(2) for c in range(2))
check("1–2. forward: the same scores as Linear", _same)
def _same_grads():
    random.seed(1); s = SparseLinear(4, 2)
    random.seed(1); d = Linear(4, 2)
    s([[0, 2], [1, 2, 3]]); d([[1, 0, 1, 0], [0, 1, 1, 1]])
    g = [[0.5, -1.0], [2.0, 0.25]]
    s.backward(g); d.backward(g)
    return s.weight.grad == d.weight.grad and s.bias.grad == d.bias.grad
check("3–4. backward: the same gradients as Linear", _same_grads)
check("it is still a Linear layer", lambda: isinstance(SparseLinear(3, 1), Linear) and len(SparseLinear(3, 1).parameters()) == 2)

---
# Part 2 · Training and evaluating on the Yelp reviews

The model is just like `LogisticRegression` from Week 8, but with a `SparseLinear` layer. Notice what is **not** in the model: the vocabulary and the tokeniser. They belong to the encoder, which turns texts into rows **before** they reach the model. Keeping the two apart means that either can be changed without touching the other (Part 4).

### 🏋️ Exercise 2 · Practice — `TextLogisticRegression`, and training

1. Complete `TextLogisticRegression`: one `SparseLinear` layer with `vocab_size` inputs and one output.
2. Make `encoder`, a `TextEncoder` with 5000 words, and build its vocabulary from **all** the texts, as the LELA60331 seminar did. Then encode the training and test texts.
3. Complete the training loop: the same five lines as in Week 8. The settings are those of the LELA60331 Week 9 seminar: `random.seed(2)` before the model is made, a learning rate of 0.4 and 500 epochs. Training takes about a minute.

In [ ]:
class TextLogisticRegression(Module):
    """Logistic regression for one-hot encoded texts."""
    def __init__(self, vocab_size):
        super().__init__()
        self.linear = ???                  # 1.

    def forward(self, M):
        return ???

    def backward(self, grad):
        return ???


encoder = ???                                        # 2.
???
M_train = ???
M_test = ???

random.seed(2)                                                       # 3.
model = TextLogisticRegression(5000)
loss_fn = BCEWithLogitsLoss()
optimizer = SGD(model.parameters(), lr=0.4)
loss_history = []
for epoch in range(500):
    ???
    loss = ???
    loss_history.append(loss)
    ???
    ???
    if epoch % 50 == 0:
        print("epoch", epoch, " loss", round(loss, 3))

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("final loss:", round(loss_history[-1], 3))          # you should get 0.598

In [ ]:
# ✅ Check
check("1. one SparseLinear layer with 5000 weights", lambda: isinstance(model.linear, SparseLinear) and len(model.linear.weight.value[0]) == 5000)
check("2. the encoded texts", lambda: len(M_train) == 8000 and len(M_test) == 2000 and len(encoder) == 5000)
check("3. the same final loss as LELA60331 Week 9 (0.598)", lambda: abs(loss_history[-1] - 0.598) < 0.001)

### 🏋️ Exercise 3 · Practice — Evaluation

`evaluate` from Week 8 works for this model too, because it only needs the model's scores.

1. `test_results` and `train_results`: the results on the test set and on the training set
2. `my_probs`: the probability that each of the reviews in `my_reviews` is positive. *Hint:* encode them first.

Can you write a review that fools the model?

In [ ]:
test_results = ???                    # 1.
train_results = ???
print("test:    ", test_results)
print("training:", train_results)

my_reviews = ["the food was amazing and the staff were friendly !",
              "terrible service , cold food . never again ."]
my_probs = ???   # 2.
for review, p in zip(my_reviews, my_probs):
    print(round(p, 3), review)

In [ ]:
# ✅ Check
check("1. test accuracy 0.82, precision 0.807, recall 0.829",
      lambda: abs(test_results["accuracy"] - 0.82) < 0.001 and abs(test_results["precision"] - 0.807) < 0.001 and abs(test_results["recall"] - 0.829) < 0.001)
check("1. training precision 0.857, recall 0.862",
      lambda: abs(train_results["precision"] - 0.857) < 0.001 and abs(train_results["recall"] - 0.862) < 0.001)
check("2. my_probs", lambda: len(my_probs) == 2 and my_probs[0] > 0.5 > my_probs[1])

These are the same numbers as in the LELA60331 Week 9 seminar. As in the seminar, the model does better on the training reviews than on the test reviews: it has **overfitted** a little.

---
# Part 3 · Inspecting the weights

Each of the model's weights belongs to one vocabulary word: `model.linear.weight.value[0][j]` is the weight for the word `encoder[j]`. Words with **high** weights push the model towards *positive*, and words with **low** weights push it towards *negative* (LELA60331 Week 9, Activity 7).

The weights live in the model and the words live in the encoder, so a function that lists the most important words needs both. Like `evaluate`, it is a function outside the classes.

### 🏋️ Exercise 4 · Practice — `top_words(model, encoder, n)`

1. Complete `top_words`, which returns two lists: the `n` words with the lowest weights, and the `n` with the highest (highest first). It sorts `(weight, word)` pairs from the lowest weight to the highest. *Hint:* the slice `[::-1]` reverses a list.
2. `lowest` and `highest`: the 15 most negative and most positive words
3. `not_words`: the tokens among the 100 lowest and 100 highest that are **not** made only of letters (`isalpha()`, Week 2)

Which words are about sentiment in general? Which suggest **overfitting** (LELA60331 Week 9, Activity 8)? Are the tokens in `not_words` useful to the model, or noise?

In [ ]:
def top_words(model, encoder, n):
    weights = model.linear.weight.value[0]
    pairs = ???     # 1.
    lowest = ???
    highest = ???
    return lowest, highest

lowest, highest = ???                       # 2.
print(lowest)
print(highest)

lowest_100, highest_100 = top_words(model, encoder, 100)
not_words = ???   # 3.
print(not_words)

In [ ]:
# ✅ Check
check("1–2. lowest", lambda: lowest[:3] == ["?", "no", "not"] and len(lowest) == 15)
check("1–2. highest", lambda: highest[:3] == ["great", "definitely", "excellent"] and len(highest) == 15)
check("3. not_words", lambda: not_words == ["?", "!"])

*Write your observations here.*

---
# Part 4 · A better tokeniser: just swap the encoder

Many of the 5000 tokens in the vocabulary aren't really words: punctuation, numbers, and `n` (what is left of the line breaks in the original reviews). Exercise 4 showed that two of them, `?` and `!`, are among the most useful tokens of all. Would the model be better without the rest?

Because the encoder and the model are separate, trying a different tokeniser needs no change to the model at all: write a new **encoder** (a subclass of `TextEncoder` that overrides `tokenise`, as with `StopwordEncoder` in Week 9), use it to encode the texts, and train a new model on its rows.

### 🏋️ Exercise 5 · Practice — `CleanEncoder`

1. Complete `tokenise` in `CleanEncoder`. It should call the parent's `tokenise` and keep only the tokens that are made of letters and are not `"n"`.
2. Make `clean_encoder` with 5000 words, build its vocabulary from all the texts, and encode the training and test texts.
3. Train `clean_model`, a new `TextLogisticRegression`, on the clean rows, with the same settings as before. *Hint:* copy your training loop. Training takes about a minute.
4. `clean_results`: its results on the test set

Is it better or worse than `model`? Look at its top words: what has changed? Can you explain the difference? (Think about Exercise 4.) Then try an encoder that keeps `!` and `?`.

In [ ]:
class CleanEncoder(TextEncoder):
    """A TextEncoder that only uses tokens made of letters."""
    def tokenise(self, text):
        return ???    # 1.

print(CleanEncoder().tokenise("i paid $ 20 for 2 pizzas ! n nthe best ."))

clean_encoder = ???                                   # 2.
???
C_train = ???
C_test = ???

random.seed(2)                                                         # 3.
clean_model = TextLogisticRegression(5000)
loss_fn = BCEWithLogitsLoss()
optimizer = SGD(clean_model.parameters(), lr=0.4)
for epoch in range(500):
    ???
    loss = ???
    ???
    ???

clean_results = ???           # 4.
print("naive tokens:", test_results)
print("clean tokens:", clean_results)
print(top_words(clean_model, clean_encoder, 15))

In [ ]:
# ✅ Check
check("1. tokenise", lambda: CleanEncoder().tokenise("i paid $ 20 for 2 pizzas ! n nthe best .") == ["i", "paid", "for", "pizzas", "nthe", "best"])
check("2. no punctuation in the clean vocabulary", lambda: clean_encoder.index(".") is None and clean_encoder.index("n") is None and len(clean_encoder) == 5000)
check("3–4. clean_results", lambda: set(clean_results) == {"accuracy", "precision", "recall"})
check("the first model is unchanged", lambda: abs(evaluate(model, M_test, test_labels)["accuracy"] - 0.82) < 0.001)

Look at the token `nthe` in the output of `tokenise`: it is a line break stuck to the next word. ⭐ **Challenge (optional):** write an encoder that also fixes tokens like this, by removing an `n` from the start of any token that isn't a word on its own. How can you tell which tokens to fix? Compare its results with the other two models.

In [ ]:
# ⭐ Challenge

---
## ✏️ Summary

You can now:

* write a subclass of a layer (`SparseLinear`) that overrides `forward` and `backward`, and check it against the original
* build a text classifier by putting together parts that each do one job: an encoder, a layer, a model, a loss and an optimiser
* reuse the same training loop and evaluation functions for a different model and different data
* keep the text processing (the encoder) separate from the model, so that either can be changed without the other
* inspect what a model has learned from its weights

Over Weeks 8, 9 and 10 you have built, from small parts, the complete review classifier from the LELA60331 seminars.